# Build & deploy the AgentCore agent

Run this notebook from the **`agent/`** folder (next to `customer_agent.py` and `requirements.txt`) in a SageMaker AI Jupyter notebook or Studio JupyterLab space.

It will:
1. Install the agent's dependencies for **linux/arm64, Python 3.12** (AgentCore Runtime runs on arm64)
2. Package `customer_agent.py` + dependencies into `agent.zip`
3. Upload the zip to S3 under a new versioned key
4. *(optional)* Update the CloudFormation stack so the runtime picks up the new code
5. *(optional)* Invoke the agent with a test prompt

**Notebook execution role needs:** `s3:PutObject` on the artifact bucket; for steps 4–5 also `cloudformation:DescribeStacks`/`UpdateStack`, `iam:PassRole` on the stack's roles (or a CloudFormation service role), and `bedrock-agentcore:InvokeAgentRuntime`.

> The SageMaker instance is usually x86_64, so the arm64 wheels can't be imported here. Step 2 checks the package contents instead of running it.

## 0. Configuration

In [ ]:
import os, sys, json, time, shutil, subprocess, zipfile, platform, py_compile
from pathlib import Path
import boto3

PROJECT        = "agentcore-poc"          # also the CloudFormation stack name
REGION         = boto3.Session().region_name or "us-east-1"
ARTIFACT_BUCKET = ""                          # leave "" to use the SageMaker default bucket
PYTHON_VERSION = "3.12"                       # must match Runtime: PYTHON_3_12 in the template
PLATFORM       = "manylinux2014_aarch64"      # AgentCore Runtime = arm64

DEPLOY_STACK   = False                        # True -> step 4 updates the stack with the new zip
RUN_TEST       = False                        # True -> step 5 invokes the agent after deploy

AGENT_DIR = Path.cwd()
BUILD_DIR = AGENT_DIR / "build"
PKG_DIR   = BUILD_DIR / "pkg"
ZIP_PATH  = BUILD_DIR / "agent.zip"

if not ARTIFACT_BUCKET:
    import sagemaker
    ARTIFACT_BUCKET = sagemaker.Session().default_bucket()

CODE_KEY = f"{PROJECT}/agent-{time.strftime('%Y%m%d%H%M%S')}.zip"

for f in ("customer_agent.py", "requirements.txt"):
    assert (AGENT_DIR / f).exists(), f"{f} not found in {AGENT_DIR} - open this notebook from the agent/ folder"

print(f"Region      : {REGION}")
print(f"Agent dir   : {AGENT_DIR}")
print(f"Upload to   : s3://{ARTIFACT_BUCKET}/{CODE_KEY}")
print(f"Notebook CPU: {platform.machine()} (target: aarch64)")

## 1. Install dependencies for linux/arm64

In [ ]:
shutil.rmtree(BUILD_DIR, ignore_errors=True)
PKG_DIR.mkdir(parents=True)

cmd = [sys.executable, "-m", "pip", "install", "--quiet",
       "-r", str(AGENT_DIR / "requirements.txt"),
       "--target", str(PKG_DIR),
       "--platform", PLATFORM,
       "--python-version", PYTHON_VERSION,
       "--implementation", "cp",
       "--only-binary=:all:",
       "--upgrade"]
print(" ".join(cmd))
res = subprocess.run(cmd, capture_output=True, text=True)
if res.returncode != 0:
    print(res.stdout[-3000:]); print(res.stderr[-3000:])
assert res.returncode == 0, "pip install failed - see output above"

# Report key package versions
VERSIONS = {}
for pkg in ("bedrock_agentcore", "strands_agents", "mcp", "httpx", "boto3"):
    hits = sorted(PKG_DIR.glob(f"{pkg}-*.dist-info"))
    VERSIONS[pkg] = hits[-1].name[len(pkg) + 1:-len(".dist-info")] if hits else None
    print(f"{pkg:18s} {VERSIONS[pkg] or 'NOT FOUND'}")

# customer_agent.py targets the mcp 1.x client API (2.x renamed it and moved to httpx2)
assert VERSIONS["mcp"] and VERSIONS["mcp"].startswith("1."), f"mcp {VERSIONS['mcp']} - customer_agent.py needs mcp 1.x"

## 2. Add the agent code and check the package

In [ ]:
# Syntax-check the agent before packaging
py_compile.compile(str(AGENT_DIR / "customer_agent.py"), doraise=True)
shutil.copy2(AGENT_DIR / "customer_agent.py", PKG_DIR / "customer_agent.py")

# Native extensions must be aarch64 builds, otherwise the runtime fails at import time
so_files = list(PKG_DIR.rglob("*.so"))
wrong = [p for p in so_files if "x86_64" in p.name]
print(f"{len(so_files)} native extension(s); x86_64 builds found: {len(wrong)}")
for p in so_files[:10]:
    print("  ", p.relative_to(PKG_DIR))
assert not wrong, f"x86_64 binaries in package: {wrong[:3]}"

# Required top-level modules for customer_agent.py imports
for mod in ("bedrock_agentcore", "strands", "mcp", "httpx", "botocore"):
    assert (PKG_DIR / mod).exists(), f"missing module '{mod}' in package"
print("Package contents OK")

## 3. Zip and upload to S3

In [ ]:
# Remove caches to keep the zip small
for p in list(PKG_DIR.rglob("__pycache__")):
    shutil.rmtree(p, ignore_errors=True)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(PKG_DIR.rglob("*")):
        if f.is_file():
            zf.write(f, f.relative_to(PKG_DIR))

with zipfile.ZipFile(ZIP_PATH) as zf:
    assert "customer_agent.py" in zf.namelist(), "customer_agent.py must be at the zip root (EntryPoint: [customer_agent.py])"
    n = len(zf.namelist())
size_mb = ZIP_PATH.stat().st_size / 1024 / 1024
print(f"{ZIP_PATH.name}: {size_mb:.1f} MB, {n} files")

s3 = boto3.client("s3", region_name=REGION)
s3.upload_file(str(ZIP_PATH), ARTIFACT_BUCKET, CODE_KEY)
print(f"Uploaded -> s3://{ARTIFACT_BUCKET}/{CODE_KEY}")
print(f"\nUse this as the stack's AgentCodeKey parameter:\n  {CODE_KEY}")

## 4. *(optional)* Update the stack with the new code

Set `DEPLOY_STACK = True` in step 0. This keeps every existing parameter and only changes `ArtifactBucket` and `AgentCodeKey`, so the stack must already exist (first deploy: `scripts/deploy.sh` or the console).

The runtime role is only allowed to read the exact key in `AgentCodeKey`, and the template updates that policy in the same change.

In [ ]:
if DEPLOY_STACK:
    cfn = boto3.client("cloudformation", region_name=REGION)
    stack = cfn.describe_stacks(StackName=PROJECT)["Stacks"][0]
    params = []
    for p in stack["Parameters"]:
        k = p["ParameterKey"]
        if k == "AgentCodeKey":
            params.append({"ParameterKey": k, "ParameterValue": CODE_KEY})
        elif k == "ArtifactBucket":
            params.append({"ParameterKey": k, "ParameterValue": ARTIFACT_BUCKET})
        else:
            params.append({"ParameterKey": k, "UsePreviousValue": True})

    cfn.update_stack(StackName=PROJECT, UsePreviousTemplate=True,
                     Parameters=params, Capabilities=["CAPABILITY_IAM"])
    print(f"Updating {PROJECT} ...")
    try:
        cfn.get_waiter("stack_update_complete").wait(
            StackName=PROJECT, WaiterConfig={"Delay": 15, "MaxAttempts": 120})
        print("Stack update complete")
    except Exception as e:
        print("Update did not complete:", e)
        for ev in cfn.describe_stack_events(StackName=PROJECT)["StackEvents"][:10]:
            if "FAILED" in ev["ResourceStatus"]:
                print(f"  {ev['LogicalResourceId']}: {ev.get('ResourceStatusReason')}")
        raise
else:
    print("Skipped (DEPLOY_STACK = False)")

## 5. *(optional)* Invoke the agent

In [ ]:
import uuid

def ask(prompt: str, session_id: str | None = None) -> str:
    cfn = boto3.client("cloudformation", region_name=REGION)
    outs = {o["OutputKey"]: o["OutputValue"]
            for o in cfn.describe_stacks(StackName=PROJECT)["Stacks"][0]["Outputs"]}
    client = boto3.client("bedrock-agentcore", region_name=REGION)
    resp = client.invoke_agent_runtime(
        agentRuntimeArn=outs["AgentRuntimeArn"],
        runtimeSessionId=session_id or f"{uuid.uuid4()}-nb",   # must be >= 33 chars
        payload=json.dumps({"prompt": prompt}).encode())
    body = resp["response"].read().decode()
    try:
        return json.loads(body)["result"]
    except (ValueError, KeyError):
        return body

if RUN_TEST:
    for q in ["List active customers in Ontario",
              "What accounts does C-1002 have and what's the total balance?"]:
        print(f">>> {q}\n{ask(q)}\n")
else:
    print("Skipped (RUN_TEST = False). You can still call ask('...') directly.")